In [1]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import pandas as pd

from src.data.market_loader import MarketLoader
from src.data.synthetic_sofr_builder import build_term_sofr_curve

from src.term_structure.bootstrap_curve import (
    bootstrap_df_from_sofr, 
    build_zero_curve_from_df,
    bootstrap_df_from_treasury_curve
)
from src.term_structure.curve_interpolation import (
    build_coupon_structure,
    zero_rate_to_df,
    interpolate_zero_curve
)

In [2]:
# downloading market curves
loader = MarketLoader()
curves = loader.loader_pipeline()

# building sythetic sofr curve from ON rates and futures
sofr_curve = build_term_sofr_curve(curves = curves)

treasury curve dataset already downloaded..
sofr curve dataset already downloaded..
futures curve dataset already downloaded..


In [3]:
# bootstrapping discount factors from sofr curve
df_bootstrap_from_sofr = bootstrap_df_from_sofr(
    sofr_curve = sofr_curve
)

df_bootstrap_from_sofr

,ON,1M,3M
2018-04-03,0.999949,0.998461,0.995322
2018-04-04,0.999952,0.998519,0.995372
2018-04-05,0.999951,0.998511,0.995322
2018-04-06,0.999951,0.998511,0.995372
2018-04-09,0.999951,0.998502,0.995297
...,...,...,...
2026-04-15,0.999897,0.996943,0.991080
2026-04-16,0.999898,0.996976,0.991129
2026-04-17,0.999899,0.996992,0.991179
2026-04-20,0.999899,0.996992,0.991105


In [4]:
# constructing zero curve from discount curve
zero_curve_mm = build_zero_curve_from_df(
    discount_curve = df_bootstrap_from_sofr
)

zero_curve_mm

,ON,1M,3M
2018-04-03,1.829953,1.848575,1.875596
2018-04-04,1.739958,1.778681,1.855689
2018-04-05,1.749957,1.788666,1.875596
2018-04-06,1.749957,1.788666,1.855689
2018-04-09,1.749957,1.798651,1.885549
...,...,...,...
2026-04-15,3.719808,3.674369,3.583897
2026-04-16,3.669813,3.634490,3.564074
2026-04-17,3.649815,3.614551,3.544251
2026-04-20,3.629817,3.614551,3.573986


In [5]:
# interpolate zero curve to coupon structure
coupon_structure = build_coupon_structure(
    max_year = 30,
    freq = 2
)

zero_curve_mm_interp = interpolate_zero_curve(
    zero_curve_df = zero_curve_mm,
    target_times = coupon_structure
)

# df bootstrap using interpolated zero curve
df_mm_interp = zero_rate_to_df(
    zero_curve_interp = zero_curve_mm_interp
)

df_mm_interp

,0.5,1.0,1.5,2.0,2.5,3.0,3.5,4.0,4.5,5.0,...,25.5,26.0,26.5,27.0,27.5,28.0,28.5,29.0,29.5,30.0
2018-04-03,0.990666,0.981419,0.972258,0.963183,0.954192,0.945286,0.936462,0.927721,0.919062,0.910483,...,0.619851,0.614065,0.608333,0.602655,0.597029,0.591457,0.585936,0.580467,0.575049,0.569681
2018-04-04,0.990764,0.981614,0.972549,0.963566,0.954667,0.945851,0.937115,0.928460,0.919886,0.911390,...,0.623005,0.617251,0.611551,0.605903,0.600307,0.594763,0.589270,0.583827,0.578436,0.573093
2018-04-05,0.990666,0.981419,0.972258,0.963183,0.954192,0.945286,0.936462,0.927721,0.919062,0.910483,...,0.619851,0.614065,0.608333,0.602655,0.597029,0.591457,0.585936,0.580467,0.575049,0.569681
2018-04-06,0.990764,0.981614,0.972549,0.963566,0.954667,0.945851,0.937115,0.928460,0.919886,0.911390,...,0.623005,0.617251,0.611551,0.605903,0.600307,0.594763,0.589270,0.583827,0.578436,0.573093
2018-04-09,0.990617,0.981321,0.972113,0.962991,0.953955,0.945004,0.936136,0.927352,0.918650,0.910030,...,0.618279,0.612478,0.606731,0.601037,0.595398,0.589811,0.584276,0.578794,0.573363,0.567983
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-04-15,0.982240,0.964796,0.947661,0.930831,0.914299,0.898061,0.882112,0.866446,0.851058,0.835943,...,0.400960,0.393839,0.386844,0.379974,0.373226,0.366597,0.360087,0.353692,0.347410,0.341240
2026-04-16,0.982337,0.964987,0.947943,0.931200,0.914752,0.898596,0.882724,0.867133,0.851817,0.836772,...,0.402992,0.395874,0.388882,0.382013,0.375266,0.368638,0.362127,0.355731,0.349447,0.343275
2026-04-17,0.982435,0.965178,0.948225,0.931569,0.915206,0.899130,0.883337,0.867821,0.852577,0.837602,...,0.405034,0.397920,0.390930,0.384063,0.377317,0.370690,0.364178,0.357781,0.351497,0.345323
2026-04-20,0.982289,0.964891,0.947802,0.931015,0.914526,0.898328,0.882418,0.866789,0.851437,0.836357,...,0.401975,0.394855,0.387862,0.380992,0.374244,0.367616,0.361105,0.354710,0.348427,0.342256


In [6]:
### constructing discount factors from tresury par yields
# get treasury par yields
treasury_curve = curves['treasury']

# bootstrapping discount factors from treasury par yields merging money market dfs
df_bootstrap_from_treasury_full = bootstrap_df_from_treasury_curve(
    treasury_curve = treasury_curve,
    money_market_dfs = df_mm_interp
)